# Prompt
Using data from anywhere on the internet, using previous 1 or 2 years data (excluding latest quarter!), build a linear [y = a+bx ], logarithmic [y = a+b*log(x) ], exponential [ y=b*exp(a*x) ], and power curve [ y=b*x^a ] models on revenue, earnings, and dividends, for symbols IBM, MSFT, AAPL, GOOG, META, PG, GE.

# Stage 1: Getting the Data
The best source right now to get the data is Yahoo Finance. Although acquiring data through the front end has a paywall system developers can build wrapper sytem that pull from the backend api. python has a library called Yfinance

In [1]:
%pip install yfinance pandas 

^C
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached websockets-16.0-cp314-cp314-win_amd64.whl.metadata (7.0 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached

# Step 2: Fetching the Raw Data
We will test the Yahoo finance to see if it pulls the appropiate data. ie Apple Statement, dividend hisory and prints them out

In [ ]:
import yfinance as yf
import pandas as pd

# Define the stock ticker 
ticker = yf.Ticker("AAPL")

# fetch the quarterly financial data Income Statement
quarterly_financials = ticker.quarterly_income_stmt

# Display the quarterly financial data
print(quarterly_financials)

#2 fetch the Dividend History
dividends = ticker.dividends

# Display the Dividend History
print(dividends.tail(10))

                                                      2025-12-31  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  1.750000e-01   
Normalized EBITDA                                   5.406600e+10   
Net Income From Continuing Operation Net Minori...  4.209700e+10   
Reconciled Depreciation                             3.214000e+09   
Reconciled Cost Of Revenue                          7.452500e+10   
EBITDA                                              5.406600e+10   
EBIT                                                5.085200e+10   
Normalized Income                                   4.209700e+10   
Net Income From Continuing And Discontinued Ope...  4.209700e+10   
Total Expenses                                      9.290400e+10   
Total Operating Income As Reported                  5.085200e+10   
Diluted Average Shares                              1.481036e+10   
Basic Average Shares                            

# Step 3: Isolate the data thats most important to Us which is revenue, Income and dividents 


In [ ]:
import numpy as np

#1). isolate the specific roles we want using the .loc method
#we want the 'Total Revenue' and 'Net Income' rows

revenue_raw = quarterly_financials.loc['Total Revenue']
earnings_raw = quarterly_financials.loc['Net Income']

#Code Borrowed from Gemini 
# 2). Reverse the order! We want oldest-to-newest for our timeline.
revenue_series = revenue_raw.iloc[::-1].tail(4)  # we only need the last 4 quarters
earnings_series = earnings_raw.iloc[::-1].tail(4)  # we only need the last 4 quarters

print(revenue_series)
print(earnings_series)

#3). Grab the last 4 dividends payouts to match our 4 quarterly_financials
dividends_series = dividends.tail(4)
print(dividends_series)


2025-03-31    9.535900e+10
2025-06-30    9.403600e+10
2025-09-30    1.024660e+11
2025-12-31    1.437560e+11
Name: Total Revenue, dtype: float64
2025-03-31    2.478000e+10
2025-06-30    2.343400e+10
2025-09-30    2.746600e+10
2025-12-31    4.209700e+10
Name: Net Income, dtype: float64
Date
2025-05-12 09:30:00-04:00    0.26
2025-08-11 09:30:00-04:00    0.26
2025-11-10 09:30:00-05:00    0.26
2026-02-09 09:30:00-05:00    0.26
Name: Dividends, dtype: float64


# Step 4: Create our X and Y axis where x is time ie (quarters) and Y is the financial Statement
Primarily we also reseve One quarters for testing purposes and one revenue column as well

In [ ]:
#4). Create Our X-axis: a timeline of the quarters

x_all = np.array([1, 2, 3, 4])  #quarters: Q1, Q2, Q3, Q4
x_train = np.array([1, 2, 3])  # training on the first 3 quarters: Q1, Q2, Q3 
x_test = np.array([4])      # Test quarter: Q4

#5). Create our y-axes: the revenue, earnings, and dividends for training
y_revenue_train = revenue_series.iloc[:3].values.astype(float)
y_earnings_train = earnings_series.iloc[:3].values.astype(float)
y_dividend_train = dividends_series.iloc[:3].values.astype(float)

# 6. Save the 4th value to check oam ur accuracy later
# actual_revenue_q4 = float(revenue_series.iloc[3])
actual_revenue_q4 = float(np.ravel(revenue_series.iloc[3])[0])

print(f"X Time Steps for training: {x_train}")
print(f"Y Revenue for training: {y_revenue_train}")
print(f"Hidden Q4 Revenue to predict later: {actual_revenue_q4}")

X Time Steps for training: [1 2 3]
Y Revenue for training: [9.53590e+10 9.40360e+10 1.02466e+11]
Hidden Q4 Revenue to predict later: 143756000000.0


# Step 5: Now we build our models, (linear, logarithmic,exponetials)
Linear model = Y = a + bx

In [ ]:
# 1. Build the Linear Model: y = a + bx
# np.polyfit returns the slope (b) and the intercept (a) in that order.
b_linear, a_linear = np.polyfit(x_train, y_revenue_train, 1)

print("--- The Model ---")
print(f"Linear Equation: y = {a_linear:,.2f} + {b_linear:,.2f}x")

--- The Model ---
Linear Equation: y = 90,180,000,000.00 + 3,553,500,000.00x


# step 6: Calculate the R squared Score

In [ ]:
#First, generate what the model *thinks* the first 3 quarters should have been
y_pred_train = a_linear + (b_linear * x_train)

# Calculate R-squared (1 - (Sum of Squared Residuals / Total Sum of Squares))
ss_res = np.sum((y_revenue_train - y_pred_train) ** 2)
ss_tot = np.sum((y_revenue_train - np.mean(y_revenue_train)) ** 2)
r2_linear = 1 - (ss_res / ss_tot)

print(f"R-squared Score: {r2_linear:.4f}")

R-squared Score: 0.6143


# The prediction

In [ ]:
# 3. The Prediction (Plugging in x = 4)
raw_prediction = a_linear + (b_linear * x_test)

# np.ravel() flattens any invisible array brackets, and [0] grabs the pure number
predicted_q4_revenue = float(np.ravel(raw_prediction)[0])

# Just in case our actual revenue was also hiding in an array, let's do the same
actual_revenue_q4 = float(np.ravel(actual_revenue_q4)[0])

# 4. The Error (Absolute difference between our prediction and reality)
error = abs(actual_revenue_q4 - predicted_q4_revenue)

print("\n--- Quarter 4 Results ---")
print(f"Predicted Revenue: {predicted_q4_revenue:,.2f}")
print(f"Actual Revenue:    {actual_revenue_q4:,.2f}")
print(f"Prediction Error:  {error:,.2f}")


--- Quarter 4 Results ---
Predicted Revenue: 104,394,000,000.00
Actual Revenue:    143,756,000,000.00
Prediction Error:  39,362,000,000.00


# Logarithmic & Exponential Model

In [ ]:
# --- LOGARITHMIC MODEL ---
# 1. Transform X
x_train_log = np.log(x_train)

# 2. Fit the model using the log of X
b_log, a_log = np.polyfit(x_train_log, y_revenue_train, 1)

print("--- APPLE REVENUE: LOGARITHMIC MODEL ---")
print(f"Equation: y = {a_log:,.2f} + {b_log:,.2f} * ln(x)")

# 3. Log R-squared & Prediction
y_pred_log_train = a_log + (b_log * x_train_log)
ss_res_log = np.sum((y_revenue_train - y_pred_log_train) ** 2)
# ss_tot is the same as the linear model: np.sum((y_revenue_train - np.mean(y_revenue_train)) ** 2)
r2_log = 1 - (ss_res_log / ss_tot)

predicted_q4_log = float(np.ravel(a_log + (b_log * np.log(x_test)))[0])
error_log = abs(actual_revenue_q4 - predicted_q4_log)

print(f"R-squared: {r2_log:.4f}")
print(f"Predicted Q4: ${predicted_q4_log:,.2f}")
print(f"Actual Revenue:    {actual_revenue_q4:,.2f}")
print(f"Error:        ${error_log:,.2f}\n")


# --- EXPONENTIAL MODEL ---
# 1. Transform Y
y_train_log = np.log(y_revenue_train)

# 2. Fit the model using the log of Y
# polyfit hands us 'a' as the slope, and 'ln(b)' as the intercept
slope, intercept = np.polyfit(x_train, y_train_log, 1)

# 3. Un-bend the math to get actual 'a' and 'b'
a_exp = slope
b_exp = np.exp(intercept)

print("--- APPLE REVENUE: EXPONENTIAL MODEL ---")
print(f"Equation: y = {b_exp:,.2f} * e^({a_exp:.4f}x)")

# 4. Exp R-squared & Prediction
y_pred_exp_train = b_exp * np.exp(a_exp * x_train)
ss_res_exp = np.sum((y_revenue_train - y_pred_exp_train) ** 2)
r2_exp = 1 - (ss_res_exp / ss_tot)

predicted_q4_exp = float(np.ravel(b_exp * np.exp(a_exp * x_test))[0])
error_exp = abs(actual_revenue_q4 - predicted_q4_exp)

print(f"R-squared: {r2_exp:.4f}")
print(f"Predicted Q4: ${predicted_q4_exp:,.2f}")
print(f"Actual Revenue:    {actual_revenue_q4:,.2f}")
print(f"Error:        ${error_exp:,.2f}")

--- 🍎 APPLE REVENUE: LOGARITHMIC MODEL ---
Equation: y = 93,962,129,669.66 + 5,566,936,389.80 * ln(x)
R-squared: 0.4653
Predicted Q4: $101,679,542,195.56
Actual Revenue:    143,756,000,000.00
Error:        $42,076,457,804.44

--- 🍎 APPLE REVENUE: EXPONENTIAL MODEL ---
Equation: y = 90,474,691,589.26 * e^(0.0359x)
R-squared: 0.6240
Predicted Q4: $104,463,194,095.06
Actual Revenue:    143,756,000,000.00
Error:        $39,292,805,904.94


# The power Curve Model

In [ ]:
# --- POWER MODEL ---
# 1. Transform both X and Y
x_train_log = np.log(x_train)
y_train_log = np.log(y_revenue_train)

# 2. Fit the model using the log of X and log of Y
# polyfit hands us 'a' as the slope, and 'ln(b)' as the intercept
a_power, ln_b = np.polyfit(x_train_log, y_train_log, 1)

# 3. Un-bend 'b' using np.exp()
b_power = np.exp(ln_b)

print("--- APPLE REVENUE: POWER MODEL ---")
print(f"Equation: y = {b_power:,.2f} * x^({a_power:.4f})")

# 4. Power R-squared & Prediction
y_pred_power_train = b_power * (x_train ** a_power)
ss_res_power = np.sum((y_revenue_train - y_pred_power_train) ** 2)
r2_power = 1 - (ss_res_power / ss_tot)

predicted_q4_power = float(np.ravel(b_power * (x_test ** a_power))[0])
error_power = abs(actual_revenue_q4 - predicted_q4_power)

print(f"R-squared: {r2_power:.4f}")
print(f"Predicted Q4: ${predicted_q4_power:,.2f}")
print(f"Error:        ${error_power:,.2f}")

--- 🍎 APPLE REVENUE: POWER MODEL ---
Equation: y = 94,007,867,415.15 * x^(0.0562)
R-squared: 0.4729
Predicted Q4: $101,626,839,076.48
Error:        $42,129,160,923.52


# Report Suggest Up until this point that The exponential model has the least amount of error and the highesr R squared (Coefficent of determination)

My best model missed Apple's actual Quarter 4 revenue by $39 Billion dollars! Why did it miss by so much? Because x=4 which represents the quarter ending December 31st. That is Apple's holiday quarter (when everyone buys new iPhones for Christmas). I assume math curves have a very hard time predicting massive, sudden seasonal spikes.

# Final Phase: The Automation Loop of Testing the Stocks
We are going to wrap your logic inside a for loop to automatically fetch the data, run all four models, and spit out the winner for every single company.

In [ ]:
import yfinance as yf
import numpy as np
import warnings

# Suppress messy math warnings from numpy just to keep our terminal clean
warnings.filterwarnings('ignore')

tickers = ["IBM", "MSFT", "AAPL", "GOOG", "META", "PG", "GE"]

print("Starting Master Analysis...\n")

for symbol in tickers:
    print(f"=========================================")
    print(f"   {symbol} FINANCIALS")
    print(f"=========================================")
    
    # 1. Fetch the data for the current company
    company = yf.Ticker(symbol)
    income = company.quarterly_income_stmt
    divs = company.dividends
    
    # We store the metrics in a dictionary to easily loop through them
    data_dict = {}
    
    # Try to grab 4 quarters of Revenue, Earnings, and Dividends
    try: data_dict["Revenue"] = income.loc['Total Revenue'].iloc[::-1].tail(4)
    except: pass
    
    try: data_dict["Earnings"] = income.loc['Net Income'].iloc[::-1].tail(4)
    except: pass
    
    try: data_dict["Dividends"] = divs.tail(4)
    except: pass

    # 2. Loop through each metric for this specific company
    for metric, series in data_dict.items():
        print(f"\n--- Metric: {metric} ---")
        
        # Safety check: Do we actually have 4 quarters of data?
        if len(series) < 4:
            print("Not enough data available.")
            continue
            
        # The Minimalist Setup
        x_train = np.array([1, 2, 3])
        x_test = 4
        y_train = series.iloc[:3].values.astype(float)
        actual_q4 = float(np.ravel(series.iloc[3])[0])
        
        # Calculate Total Sum of Squares (used for all R-squared calculations)
        ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
        
        # --- THE BASELINE: LINEAR MODEL ---
        b_lin, a_lin = np.polyfit(x_train, y_train, 1)
        r2_lin = 1 - (np.sum((y_train - (a_lin + (b_lin * x_train))) ** 2) / ss_tot)
        
        # Set Linear as the default "winner"
        best_model = "Linear"
        best_r2 = r2_lin
        best_pred = float(np.ravel(a_lin + (b_lin * x_test))[0])
        
        # --- THE CURVES (Only if data is purely positive!) ---
        if np.min(y_train) > 0:
            
            # Logarithmic
            x_log = np.log(x_train)
            b_log, a_log = np.polyfit(x_log, y_train, 1)
            r2_log = 1 - (np.sum((y_train - (a_log + b_log * x_log)) ** 2) / ss_tot)
            
            if r2_log > best_r2:
                best_model, best_r2 = "Logarithmic", r2_log
                best_pred = float(np.ravel(a_log + b_log * np.log(x_test))[0])
            
            # Exponential
            y_log = np.log(y_train)
            a_exp_raw, ln_b = np.polyfit(x_train, y_log, 1)
            b_exp = np.exp(ln_b)
            r2_exp = 1 - (np.sum((y_train - (b_exp * np.exp(a_exp_raw * x_train))) ** 2) / ss_tot)
            
            if r2_exp > best_r2:
                best_model, best_r2 = "Exponential", r2_exp
                best_pred = float(np.ravel(b_exp * np.exp(a_exp_raw * x_test))[0])
                
            # Power
            a_pow, ln_b_pow = np.polyfit(x_log, y_log, 1)
            b_pow = np.exp(ln_b_pow)
            r2_pow = 1 - (np.sum((y_train - (b_pow * (x_train ** a_pow))) ** 2) / ss_tot)
            
            if r2_pow > best_r2:
                best_model, best_r2 = "Power", r2_pow
                best_pred = float(np.ravel(b_pow * (x_test ** a_pow))[0])
                
        # 3. Print the results for the best model
        error = abs(actual_q4 - best_pred)
        print(f"Winning Model: {best_model} (R-squared: {best_r2:.4f})")
        print(f"Predicted Q4:  ${best_pred:,.2f}")
        print(f"Actual Q4:     ${actual_q4:,.2f}")
        print(f"Error Margin:  ${error:,.2f}")

print("\nAnalysis Complete!")

Starting Master Analysis...

   IBM FINANCIALS

--- Metric: Revenue ---
Winning Model: Logarithmic (R-squared: 0.6506)
Predicted Q4:  $17,395,440,386.13
Actual Q4:     $19,687,000,000.00
Error Margin:  $2,291,559,613.87

--- Metric: Earnings ---
Winning Model: Logarithmic (R-squared: 0.5088)
Predicted Q4:  $2,245,515,703.66
Actual Q4:     $5,600,000,000.00
Error Margin:  $3,354,484,296.34

--- Metric: Dividends ---
Winning Model: Linear (R-squared: -inf)
Predicted Q4:  $1.68
Actual Q4:     $1.68
Error Margin:  $0.00
   MSFT FINANCIALS

--- Metric: Revenue ---
Winning Model: Logarithmic (R-squared: 0.9515)
Predicted Q4:  $80,383,263,694.29
Actual Q4:     $81,273,000,000.00
Error Margin:  $889,736,305.71

--- Metric: Earnings ---
Winning Model: Logarithmic (R-squared: 0.9874)
Predicted Q4:  $28,339,788,021.89
Actual Q4:     $38,458,000,000.00
Error Margin:  $10,118,211,978.11

--- Metric: Dividends ---
Winning Model: Exponential (R-squared: 0.7610)
Predicted Q4:  $0.94
Actual Q4:     $0.